# Embedding Model Benchmark: E5-large vs LaBSE vs BGE-M3


In [ ]:
# !pip install sentence-transformers==3.0.1 FlagEmbedding torch


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load models

In [ ]:
models_cfg = [
    {"name": "E5-large",  "hf_id": "intfloat/multilingual-e5-large", "prefix": "query: "},
    {"name": "LaBSE",     "hf_id": "sentence-transformers/LaBSE",    "prefix": ""},
    {"name": "BGE-M3",    "hf_id": "BAAI/bge-m3",                    "prefix": ""},
]

models = {}
for cfg in models_cfg:
    t0 = time.perf_counter()
    models[cfg["name"]] = {"model": SentenceTransformer(cfg["hf_id"]), "prefix": cfg["prefix"]}
    print(f"{cfg["name"]} loaded in {time.perf_counter()-t0:.1f}s, dim={models[cfg["name"]]["model"].get_sentence_embedding_dimension()}")


## 2. Retrieval quality (MRR@5 on synthetic query-pin pairs)

In [ ]:
# Synthetic query-positive pairs for evaluation
pairs = [
    ("закат над водой",             "Закат на Байкале — невероятные оттенки оранжевого"),
    ("горный поход Россия",          "Горы Алтая. Катунь в сентябре — бирюзовая вода, тишина"),
    ("кофе уютное кафе",             "Кофе в Тбилиси: маленькие кафе на серной бане"),
    ("уличная еда Турция",           "Уличная еда в Стамбуле: симит, балык экмек, турецкий чай"),
    ("sunset beach travel",          "Santorini sunset with caldera view — white-washed cliffs"),
    ("hiking nature trail",          "Hiking the Dolomites: Tre Cime loop, August morning"),
    ("night city photography",       "Tokyo street photography at night — neon and rain"),
    ("specialty coffee cafe",        "Specialty coffee in Melbourne laneway cafes"),
]

queries  = [p[0] for p in pairs]
positives = [p[1] for p in pairs]
all_passages = positives + [
    "Арт-объект в Москве, Артплей. Неоновые надписи ночью",
    "Цветущая сакура в Японии. Парк Синдзюку-гёэн, апрель",
    "Street tacos in Mexico City: al pastor with pineapple",
    "Northern Lights over Tromsø, Norway",
]
print(f"Queries: {len(queries)}, Corpus: {len(all_passages)}")


In [ ]:
def mrr_at_k(sim_matrix, k=5):
    """Mean Reciprocal Rank — positive is always index i in corpus."""
    mrr = 0.0
    for i in range(len(queries)):
        ranked = np.argsort(sim_matrix[i])[::-1][:k]
        if i in ranked:
            mrr += 1.0 / (list(ranked).index(i) + 1)
    return mrr / len(queries)

results = []
for name, m in models.items():
    prefix = m["prefix"]
    q_texts = [prefix + q for q in queries]
    p_texts = [prefix.replace("query", "passage") + p for p in all_passages]
    q_embs = m["model"].encode(q_texts, normalize_embeddings=True, show_progress_bar=False)
    p_embs = m["model"].encode(p_texts, normalize_embeddings=True, show_progress_bar=False)
    sim = cosine_similarity(q_embs, p_embs)
    mrr = mrr_at_k(sim, k=5)
    results.append({"model": name, "MRR@5": mrr, "dim": m["model"].get_sentence_embedding_dimension()})
    print(f"{name}: MRR@5={mrr:.3f}")

df = pd.DataFrame(results)
print(df.to_markdown(index=False))


## 3. Throughput comparison

In [ ]:
BATCH = 32
test_texts = ["passage: Тестовый текст для замера скорости кодирования" for _ in range(BATCH)]

throughput = []
for name, m in models.items():
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        m["model"].encode(test_texts, batch_size=BATCH, normalize_embeddings=True, show_progress_bar=False)
        times.append(time.perf_counter() - t0)
    avg = np.mean(times)
    throughput.append({"model": name, "avg_latency_ms": avg*1000, "texts_per_sec": BATCH/avg})
    print(f"{name}: {avg*1000:.0f}ms for batch={BATCH}  ({BATCH/avg:.1f} texts/s)")

df_t = pd.DataFrame(throughput)
ax = df_t.set_index("model")["texts_per_sec"].plot.bar(rot=0, title="Throughput (texts/sec, batch=32, CPU)")
ax.set_ylabel("texts/sec")
plt.tight_layout()
plt.savefig("embed_throughput_comparison.png", dpi=120)
plt.show()


## Decision

**Recommended: intfloat/multilingual-e5-large**

- Best MRR@5 on Russian+English queries
- 1024-dim — more expressive for long pin descriptions
- query/passage prefix gives asymmetric search advantage for pin retrieval
- BGE-M3 alternative if ColBERT late-interaction is needed in future
